# RAPTOR — 계층적 요약으로 긴 문서를 정복하는 RAG

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **RAPTOR**의 계층적 요약 구조와 왜 긴 문서에 효과적인지 이해하기
- 원본 청크(레벨 0)와 그룹 요약(레벨 1)을 하나의 벡터스토어에 통합하기
- 질문의 추상화 수준에 따라 적절한 레벨에서 검색이 이루어지는 과정 확인하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- LLM Chain으로 문서 요약하는 방법

---

```mermaid
flowchart TD
    D[원본 문서]:::input --> L0[레벨 0<br/>원본 청크 800자]:::process
    L0 --> G[5개씩 그룹화]:::process
    G --> L1[레벨 1<br/>그룹 요약]:::process
    L0 --> VS[(통합 벡터스토어<br/>레벨 0 + 레벨 1)]:::storage
    L1 --> VS
    Q[사용자 질문]:::input --> VS
    VS --> R[레벨별 검색<br/>k=5]:::process
    R --> A[계층적 맥락<br/>답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 왜 RAPTOR가 필요한가?

| 질문 유형 | 일반 RAG | RAPTOR |
|---------|---------|--------|
| "이 논문의 주요 기여는?" | 청크 단위로 파악 어려움 | 레벨 1 요약에서 전체 파악 |
| "3장의 실험 설정은?" | 원본 청크에서 정확히 검색 | 레벨 0에서 세부 검색 |

> **실무 팁**: 이 노트북은 RAPTOR의 핵심 아이디어를 2레벨로 간소화한 버전이에요. 실제 RAPTOR 논문은 가우시안 혼합 모델 클러스터링을 활용한 3~4 레벨 구조를 사용합니다.

## 왜 RAPTOR가 필요한가?

### 일반 RAG의 한계

긴 문서(예: 100페이지 논문)에서:
- "이 논문의 주요 기여는?" → 여러 청크를 종합해야 답변 가능
- 하지만 청크는 독립적으로 검색됨
- 전체적인 맥락 파악 어려움

### RAPTOR의 해결책

- 여러 레벨의 요약 생성
- 질문에 따라 적절한 추상화 레벨 선택
- 전체 맥락과 세부 정보 모두 활용

## 학습 목표

이 노트북에서는 **간소화된 RAPTOR**를 구현합니다:
- 2레벨 계층 구조 (원본 + 요약)
- 통합 검색 (모든 레벨에서 동시 검색)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

## 1. 긴 문서 로드

In [3]:
# 문서 로드
FILE_PATH = "./data/2026_gov.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

# 원본 청크로 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
level0_chunks = text_splitter.split_documents(docs[41:95])

In [4]:
# ---------------------------------------------------
# 청크 그룹화 후 LLM으로 레벨 1 요약 생성
# ---------------------------------------------------
# ============================================================
# TODO: 5개 청크씩 묶어 요약 체인으로 레벨 1 요약을 만드세요
# 힌트: for i in range(0, len(level0_chunks), 5) → summary_chain.invoke(combined_text)
# 예상 결과: 각 그룹의 요약 완료 메시지 출력
# ============================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

summary_prompt = PromptTemplate.from_template("""
다음 텍스트들을 하나의 일관된 요약으로 통합하세요.
핵심 개념과 주요 정보만 포함하세요.

텍스트들:
{text}

통합요약:
""")

summary_chain = summary_prompt | llm | StrOutputParser()

# 너무 크면 요약 품질 저하, 너무 작으면 레벨 1 문서 수가 많아져 이점이 줄어듦
group_size = 5
level1_summaries = []

for i in range(0, len(level0_chunks), group_size):
    group = level0_chunks[i:i + group_size]
    combined_text = "\n\n".join([chunk.page_content for chunk in group])

    summary = summary_chain.invoke({"text": combined_text})
    print(f"{i} 번째 그룹 요약: {summary}")

    # 💡 팁: metadata의 level 필드로 나중에 검색 결과가 어느 레벨에서 왔는지 추적 가능
    summary_doc = Document(
        page_content=summary,
        metadata={
            "level": 1,
            "groud_id": i // group_size,
            "source": "summary"
        }
    )

    level1_summaries.append(summary_doc)
    print(f"그룹 {i // group_size + 1} 요약 완료")



0 번째 그룹 요약: 2026년 1월 1일부터 시행되는 주요 금융·재정·조세 변화는 다음과 같습니다:

1. **통합고용세액공제 개편**: 공제액 구조가 개편되고 사후관리가 합리화되어 장기고용 유인이 강화됩니다.

2. **웹툰콘텐츠 제작비용 세액공제 신설**: 웹툰 제작에 소요되는 비용의 10% (중소기업은 15%)가 소득세 및 법인세에서 공제됩니다.

3. **보육수당 비과세 한도 확대**: 근로자 1인당 월 20만원의 보육수당 비과세 한도가 자녀 1인당으로 확대됩니다.

4. **초등 저학년 예체능 학원비 세제지원**: 초등학교 저학년 자녀의 예체능 학원비가 교육비 세액공제 대상에 포함됩니다.

5. **신용카드 소득공제 한도 확대**: 신용카드 등 소득공제 기본한도가 자녀당 50만원(최대 100만원)으로 상향 조정됩니다.

6. **고배당 기업 배당소득 분리과세 도입**: 고배당 기업의 배당소득에 대해 종합소득 과세에서 제외하고 분리과세를 허용합니다.

7. **증권거래세율 조정**: 코스피와 코스닥·K-OTC 시장의 증권거래세율이 조정됩니다.

8. **상호금융 비과세 적용기한 연장**: 농협, 수협, 산림조합 조합원에 대한 비과세 적용기한이 3년 연장됩니다.

이러한 변화들은 세액공제 및 비과세 혜택을 통해 근로자와 기업의 부담을 경감하고, 특정 산업의 성장을 지원하는 방향으로 진행됩니다.
그룹 1 요약 완료
5 번째 그룹 요약: 2026년부터 시행되는 주요 세제 개편 사항은 다음과 같습니다:

1. **보세공장 제품 과세 방식**: 보세공장에서 생산된 제품의 과세 방식 신청기한이 수입신고 전까지로 확대되며, 승인 시 기업은 유리한 과세 방식을 적용받을 수 있습니다. (시행일: 2026년 1월 1일)

2. **담배 정의 확대**: 담배의 정의가 연초의 잎에서 연초·니코틴 기반 제품으로 확대되어 관련 법령의 규제 및 과세 대상이 증가합니다. 제조업 허가와 담배소매인 지정이 필요하며, 미성년자 및 온라인 판매가 금지됩니다. (시행일: 2026년 4월 

In [10]:
# ---------------------------------------------------
# 레벨 0 + 레벨 1을 하나의 FAISS 벡터스토어로 통합
# ---------------------------------------------------
# ============================================================
# TODO: level0_chunks와 level1_summaries를 합쳐 FAISS 벡터스토어를 만드세요
# 힌트: all_documents = level0_chunks + level1_summaries, FAISS.from_documents(...)
# 예상 결과: 계층적 벡터스토어 구축 완료 (레벨별 문서 수 출력)
# ============================================================

# len(level1_summaries)
# level1_summaries[0].page_content

# 레벨 0에 메타데이터 추가
for chunk in level0_chunks:
    chunk.metadata["level"] = 0

all_documents = level0_chunks + level1_summaries
# print(f' ==> [Line 17]: \033[38;2;64;52;235m[all_documents]\033[0m({type(all_documents).__name__}) = \033[38;2;227;173;105m{all_documents}\033[0m')

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 모든 레벨 통합 — 검색 시 자동으로 가장 적합한 레벨 선택

# 벡터스토어 생성
vectorstore = FAISS.from_documents(all_documents, embedding=embeddings)

# 추상적 질문 → 요약 벡터와 높은 유사도 / 세부 질문 → 원본 청크와 높은 유사도

print(":white_check_mark: 계층적 벡터스토어 구축 완료")
print(f"- 레벨 0 (원본): {len(level0_chunks)}개")
print(f"- 레벨 1 (요약): {len(level1_summaries)}개")
print(f"- 총 문서: {len(all_documents)}개")


:white_check_mark: 계층적 벡터스토어 구축 완료
- 레벨 0 (원본): 64개
- 레벨 1 (요약): 13개
- 총 문서: 77개


## 4. RAPTOR RAG 체인

In [11]:
# ---------------------------------------------------
# RAPTOR RAG 체인 구성 — 계층적 검색 + LLM 답변
# ---------------------------------------------------
# ============================================================
# TODO: retriever와 raptor_prompt를 LCEL로 연결해 RAG 체인을 만드세요
# 힌트: search_kwargs={"k": 5}로 다양한 레벨에서 검색
# 예상 결과: "RAPTOR RAG 체인 생성 완료" 출력
# ============================================================

# Retriever — 모든 레벨(0·1)에서 동시 검색
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# 프롬프트
raptor_prompt = PromptTemplate.from_template("""
문맥을 참고하여 질문에 답하세요.
문맥은 서로 다른 추상화 레벨에서 검색된 정보입니다.

문맥:
{context}
질문:
{question}
답변:
""")

# RAPTOR RAG 체인
raptor_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | raptor_prompt
    | llm
    | StrOutputParser()
)


## 5. 테스트: 추상화 레벨별 질문

In [12]:
# 질문 1: 개괄적 질문 (레벨 1 요약에서 답변 가능)
question1 = "이 문서의 전반적인 주제는 무엇인가요?"

# 검색된 문서 확인
print(f"질문: {question1}")
print("=" * 60)

# 검색된 문서 확인
retrieved = retriever.invoke(question1)
print("\n검색된 문서 레벨:")
for i, doc in enumerate(retrieved[:3], 1):
    level = doc.metadata.get('level', 0)
    print(f"  [{i}] 레벨 {level}: {doc.page_content[:80]}...")

print("\n답변:")
answer1 = raptor_chain.invoke(question1)
print(answer1)
print()
# 아래에 코드를 작성하세요


질문: 이 문서의 전반적인 주제는 무엇인가요?

검색된 문서 레벨:
  [1] 레벨 1: 2026년부터 시행되는 주요 세제 개편 내용은 다음과 같습니다:

1. **지방이전 기업 세제지원 제도 개선**: 지방이전 기업의 세제 지원을 ...
  [2] 레벨 0: 2
금융·재정·조세
제 1 장
통합고용세액공제 공제액 구조 개편 및 사후관리 합리화
시행일: 2026년 1월 1일
자세한 내용은p.008
• ...
  [3] 레벨 0: 20
영상콘텐츠 세제지원 확대
재정경제부
www.moef.go.kr
영상콘텐츠산업의 글로벌 경쟁력 지원을 위하여 영상콘텐츠 제작비용 세액공제의 ...

답변:
이 문서의 전반적인 주제는 2026년부터 시행되는 주요 세제 개편 내용과 관련된 정보입니다. 문서에서는 지방이전 기업에 대한 세제 지원, 장애인 표준사업장 세액감면 확대, 육아휴직급여 비과세 대상 확대, 대학생 교육비 세액공제 소득요건 폐지 등 다양한 세제 변화와 그 목적을 설명하고 있습니다. 이러한 변화들은 지역균형발전, 장애인 일자리 창출, 자녀 양육 부담 완화 및 교육비 부담 경감을 목표로 하고 있습니다.



In [13]:
# 질문 2: 세부적 질문 (레벨 0 원본에서 답변)
question2 = "영상콘텐츠 제작비용 세액공제의 구체적인 공제율은?"

print(f"질문: {question2}")
print("=" * 60)

# 검색된 문서 확인
retrieved = retriever.invoke(question2)
print("\n검색된 문서 레벨:")
for i, doc in enumerate(retrieved[:3], 1):
    level = doc.metadata.get('level', 0)
    print(f"  [{i}] 레벨 {level}: {doc.page_content[:80]}...")

print("\n답변:")
answer2 = raptor_chain.invoke(question2)
print(answer2)


질문: 영상콘텐츠 제작비용 세액공제의 구체적인 공제율은?

검색된 문서 레벨:
  [1] 레벨 0: 20
영상콘텐츠 세제지원 확대
재정경제부
www.moef.go.kr
영상콘텐츠산업의 글로벌 경쟁력 지원을 위하여 영상콘텐츠 제작비용 세액공제의 ...
  [2] 레벨 0: 2
금융·재정·조세
제 1 장
통합고용세액공제 공제액 구조 개편 및 사후관리 합리화
시행일: 2026년 1월 1일
자세한 내용은p.008
• ...
  [3] 레벨 0: 9
2026년부터 이렇게 달라집니다
웹툰콘텐츠 제작비용 세액공제 신설 
재정경제부
www.moef.go.kr
웹툰콘텐츠산업의 글로벌 경쟁력 지원...

답변:
영상콘텐츠 제작비용 세액공제의 구체적인 공제율은 다음과 같습니다:

- 대기업 및 중견기업: 기본공제율 10%
- 중소기업: 기본공제율 15%

추가공제율은 대기업 및 중견기업 10%, 중소기업 15%로 동일합니다.


## 💡 핵심 정리

### RAPTOR의 핵심 아이디어

1. **계층적 요약**: 문서를 여러 추상화 레벨로 표현
2. **통합 검색**: 모든 레벨에서 동시 검색
3. **유연한 답변**: 질문의 특성에 따라 적절한 레벨 활용

### 일반 RAG vs RAPTOR

| 특징 | 일반 RAG | RAPTOR |
|------|---------|--------|
| 검색 대상 | 원본 청크만 | 원본 + 여러 레벨 요약 |
| 개괄적 질문 | 어려움 | 상위 요약에서 답변 |
| 세부적 질문 | 가능 | 원본 청크에서 답변 |
| 긴 문서 처리 | 제한적 | 효과적 |

### 실전 활용

- **연구 논문 분석**: 전체 내용 파악 + 세부 실험 질의
- **긴 보고서**: 요약 + 상세 데이터 검색
- **법률 문서**: 전체 맥락 + 특정 조항 검색

### 확장 가능성

이 노트북은 2레벨 구조를 구현했지만, 실제 RAPTOR는:
- 3~4개 레벨 사용 가능
- 클러스터링으로 관련 청크 그룹화
- 트리 구조로 계층 관리

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 원본 청크(레벨 0) + 그룹 요약(레벨 1)을 하나의 벡터스토어에 통합 |
| 검색 장점 | 세부 내용 질문엔 레벨 0, 전체 개요 질문엔 레벨 1 문서가 검색됨 |
| 이 노트북의 범위 | 간소화 버전 — 클러스터링·다단계 요약은 원논문 참조 |
| 구현 핵심 | `sum_docs` 리스트에 원본+요약 모두 추가 후 `FAISS.from_documents()` |
| 주의 | 요약 품질이 LLM 성능에 크게 의존 — 요약 프롬프트 튜닝이 중요 |

### RAPTOR 레벨 구조

| 레벨 | 내용 | 적합한 질문 유형 |
|------|------|-----------------|
| 레벨 0 | 원본 청크 (세부 정보) | "3장 2절의 구체적 내용은?" |
| 레벨 1 | 그룹 요약 (핵심 정리) | "이 문서의 전체 주제는?" |
| 레벨 2+ | 상위 요약 (원논문) | 더 넓은 추상화 — 이 노트북 범위 외 |

### 다음 노트북 예고

**05-Web-Summarization** — Map-Reduce 패턴으로 웹 문서를 대규모로 요약하고, 원본과 요약 모두를 벡터스토어에 저장해 질문 유형에 따라 최적 문서를 반환하는 방법을 배워요.